In [1]:
import random
from statistics import mean, pstdev

# Shared Constants
GAMMA = 0.9
STATES = ["A", "B"]
ACTIONS = [0, 1]

def step(state, action):
    """Returns (next_state, reward, done)."""
    if state == "A":
        return ("B", 1.0, False) if action == 0 else ("T", 0.0, True)
    if state == "B":
        return ("A", 0.0, False) if action == 0 else ("T", 2.0, True)
    return ("T", 0.0, True)

def policy_probs(state):
    """Returns stochastic policy dict: action -> prob."""
    if state == "A":
        return {0: 0.7, 1: 0.3}
    if state == "B":
        return {0: 0.4, 1: 0.6}
    return {}

def sample_action(probs):
    """Samples an action based on probability distribution."""
    x = random.random()
    cum = 0.0
    for a, p in probs.items():
        cum += p
        if x <= cum:
            return a
    return list(probs.keys())[0] # fallback

In [2]:
def run_episode(max_steps=100):
    s = "A"
    rewards = []
    traj = []
    
    for _ in range(max_steps):
        a = sample_action(policy_probs(s))
        s_next, r, done = step(s, a)
        traj.append((s, a, r, s_next, done))
        rewards.append(r)
        s = s_next
        if done:
            break
            
    # Compute discounted return G_0
    G = sum((GAMMA ** t) * r for t, r in enumerate(rewards))
    return G, traj

print("--- Question 1: Monte Carlo ---")
N = 2000
returns = [run_episode()[0] for _ in range(N)]
print(f"Mean return: {mean(returns):.4f}")
print(f"Std dev (population): {pstdev(returns):.4f}")

--- Question 1: Monte Carlo ---
Mean return: 1.8449
Std dev (population): 1.3849


In [3]:
def policy_evaluation(tol=1e-10, max_iters=10000):
    V = {"A": 0.0, "B": 0.0, "T": 0.0}
    
    for _ in range(max_iters):
        delta = 0.0
        for s in ["A", "B"]:
            v_old = V[s]
            v_new = 0.0
            for a, p in policy_probs(s).items():
                s_next, r, _ = step(s, a)
                v_new += p * (r + GAMMA * V[s_next])
            V[s] = v_new
            delta = max(delta, abs(v_old - v_new))
            
        if delta < tol:
            break
            
    return V

print("--- Question 2: Bellman Expectation Equations ---")
V_pi = policy_evaluation()
print(f"V^pi(A) = {V_pi['A']:.4f}")
print(f"V^pi(B) = {V_pi['B']:.4f}")

--- Question 2: Bellman Expectation Equations ---
V^pi(A) = 1.8831
V^pi(B) = 1.8779


In [4]:
def max_q_next(Q, s_next):
    if s_next == "T":
        return 0.0
    return max(Q[(s_next, a)] for a in ACTIONS)

def bellman_opt_backup(Q, s, a):
    s_next, r, _ = step(s, a)
    return r + GAMMA * max_q_next(Q, s_next)

def greedy_action(Q, s):
    vals = [Q[(s, a)] for a in ACTIONS]
    m = max(vals)
    best = [a for a in ACTIONS if Q[(s, a)] == m]
    return random.choice(best)

print("--- Question 3: Bellman Optimality Backups ---")
K = 6
Q = {(s, a): 0.0 for s in STATES for a in ACTIONS}

for k in range(1, K + 1):
    Q_new = Q.copy()
    for s in STATES:
        for a in ACTIONS:
            Q_new[(s, a)] = bellman_opt_backup(Q, s, a)
    Q = Q_new
    
    print(f"Sweep {k}: greedy(A)={greedy_action(Q, 'A')}, greedy(B)={greedy_action(Q, 'B')}")
    print(f"  Q(A,0)={Q[('A',0)]:.4f}, Q(A,1)={Q[('A',1)]:.4f}")
    print(f"  Q(B,0)={Q[('B',0)]:.4f}, Q(B,1)={Q[('B',1)]:.4f}")

--- Question 3: Bellman Optimality Backups ---
Sweep 1: greedy(A)=0, greedy(B)=1
  Q(A,0)=1.0000, Q(A,1)=0.0000
  Q(B,0)=0.0000, Q(B,1)=2.0000
Sweep 2: greedy(A)=0, greedy(B)=1
  Q(A,0)=2.8000, Q(A,1)=0.0000
  Q(B,0)=0.9000, Q(B,1)=2.0000
Sweep 3: greedy(A)=0, greedy(B)=0
  Q(A,0)=2.8000, Q(A,1)=0.0000
  Q(B,0)=2.5200, Q(B,1)=2.0000
Sweep 4: greedy(A)=0, greedy(B)=0
  Q(A,0)=3.2680, Q(A,1)=0.0000
  Q(B,0)=2.5200, Q(B,1)=2.0000
Sweep 5: greedy(A)=0, greedy(B)=0
  Q(A,0)=3.2680, Q(A,1)=0.0000
  Q(B,0)=2.9412, Q(B,1)=2.0000
Sweep 6: greedy(A)=0, greedy(B)=0
  Q(A,0)=3.6471, Q(A,1)=0.0000
  Q(B,0)=2.9412, Q(B,1)=2.0000


outcomes:
action values steadily increase.
optimal policy quickly adapts.
expectation evaluates fixed policy.
optimality finds maximum returns.

Short Answers:
1. Ans: The discount factor determines the current value of future rewards. Also, representing how much agent cares about the immediate glatification versus long term gains. a value closer to zero makes the agent short sighted while a value closer to 1 makes it strive for ;long term cumukative rewards

2. Ans: exploration is necessary, because of an agent initially lacks knowledge about the environment, it must try sub optimal or unknown actions to discover some potentially higher yeilding reward paths. 

3. Ans: the bellman expectation equation computes the average value of a state when following a specific, fixed policy. while the optimality equation computes value of a state under the best possible policy, mapping out the maximum achievable rewards - regard;less the current policy.